# Spectrometer design walkthrough

The same calculations the app runs, step by step. Physics lives in `optics.py`, ray tracing in `raytracing_v.py` / `grating.py`, drawing in `plot.py`.

In [ ]:
import numpy as np

from optics import (Spec, analyze, recommend_groove_density, grating_angles,
                    angular_dispersion, blaze_angle, detector_signal, line_dip, HG_AR_LINES)
from grating import trace, sensor_hits
from plot import ray_figure, detector_figure

## 1. Requirements

Wavelength span, target resolution, detector and focal lengths. `included_deg=None` lets the design pick the included angle.

In [ ]:
spec = Spec(wl_min=400, wl_max=700, resolution=1.0, pixel_um=5.0, n_pixels=2048,
            f1=50, f2=50, transmissive=False)   # Czerny–Turner
spec

## 2. Grating equation by hand

$m\lambda G = \sin\alpha + \sin\beta$, with the included angle $2K = \alpha - \beta$ fixing the layout.

In [ ]:
G, included = 300, 40.0
alpha, beta_c = grating_angles(spec.wl_center, G, included)
D = angular_dispersion(G, beta_c)          # rad/nm
print(f'α = {alpha:.2f}°, β(centre) = {beta_c:.2f}°, dispersion = {np.degrees(D) * 1e3:.2f} mdeg/nm')
print(f'blaze angle for 482 nm: {blaze_angle(482, G, included):.2f}°')

## 3. Full design and the recommended grating

`analyze` returns every derived quantity plus warnings, each with one-click fixes the app shows as buttons.

In [ ]:
design = recommend_groove_density(spec)
for k in ['G', 'included_deg', 'resolution', 'slit_um', 'spectrum_mm', 'rld', 'fold1', 'fold2']:
    print(f'{k:>14}: {getattr(design, k):.3f}')
for w in design.warnings:
    print('⚠', w, w.fixes)

## 4. Trying other gratings

In [ ]:
for G in (150, 300, 600, 1200):
    d = analyze(spec, G)
    print(f'{G:5d} l/mm  res {d.resolution:6.3f} nm  spectrum {d.spectrum_mm:6.2f} mm '
          f'(sensor {spec.sensor_mm:.2f})  warnings {len(d.warnings)}')

## 5. Ray trace

The tracer lands each wavelength on the sensor; its positions agree with the analytic spectrum length.

In [ ]:
wls = np.array([spec.wl_min, spec.wl_max])
components, bundles = trace(spec, design, wls)
hits = np.nanmean(sensor_hits(components, bundles).reshape(len(wls), -1), axis=1)
print(f'traced span {abs(hits[1] - hits[0]):.3f} mm, analytic {design.spectrum_mm:.3f} mm')

In [ ]:
ray_figure(spec, design)

## 6. What the detector sees

An Hg–Ar lamp through this design. The 577/579 nm doublet tests the resolution.

In [ ]:
lines = [l for l in HG_AR_LINES if spec.wl_min <= l <= spec.wl_max]
wl_pix, signal = detector_signal(spec, design, lines)
print(f'577/579 dip: {line_dip(wl_pix, signal, 576.96, 579.07):.0%} (≤ 81 % = resolved)')
detector_figure(wl_pix, signal, lines, (spec.wl_min, spec.wl_max))